# Recherche d’hyperparamètres PPO / SAC

Ce notebook complète les notebooks d’entraînement existants.
Installer l’extension depuis la racine du dépôt avec `pip install -e ".[hpo]"`.
Utiliser le kernel **Python (so101-robotic-learning)**.

Méthodes : `grid` teste une grille, `tpe` adapte les propositions aux résultats,
`gp` (ou `bayesian`) utilise un processus gaussien.
Chaque essai entraîne un nouveau modèle. La tâche, la reward et la seed restent fixes.
Le score est la moyenne du maximum de rebonds consécutifs sur les épisodes d’évaluation.
Configurer le backend graphique adapté à la machine avant les imports MuJoCo si nécessaire.


In [ ]:
from datetime import datetime
from pathlib import Path
import copy
from IPython.display import JSON, display

root = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "pyproject.toml").is_file()
             and (path / "src/so101_robotic_learning").is_dir()), None)
if root is None:
    raise RuntimeError("Ouvrir le notebook depuis le dépôt ou un de ses sous-dossiers.")

from so101_robotic_learning.bounce_rl.env.bounce_events import BounceConfig
from so101_robotic_learning.bounce_rl.rewards import make_default_reward
from so101_robotic_learning.bounce_rl.rl.hyperparameter_search import (
    optimize, PPO_SPACE, SAC_SPACE, PPO_GRID, SAC_GRID,
)


## Configuration

Changer `ALGORITHM` et `METHOD` avant d’exécuter la cellule.
`N_TRIALS=None` parcourt toute la grille restante, ou lance 10 nouveaux essais TPE/GP.
Une grille PPO par défaut représente 24 entraînements et la grille SAC en représente 16.
Le budget est arrondi vers le haut à une frontière de collecte commune à tous les essais.
Les essais sont séquentiels. Les GIF et les exports de replay buffers sont désactivés pendant la recherche.

Pour continuer une étude après un arrêt, réutiliser exactement son `STUDY_NAME` et les mêmes
paramètres de tâche et de recherche. Les résultats déjà terminés restent dans SQLite.
`N_TRIALS` indique un nombre d’essais supplémentaires, dans la limite de la grille restante.
Une interruption au milieu d’un essai ne reprend pas son entraînement.


In [ ]:
ALGORITHM = "ppo"  # "ppo" or "sac"
METHOD = "tpe"     # "grid", "tpe", "gp" (alias "bayesian")
STUDY_NAME = f"{ALGORITHM}_{METHOD}_search_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
N_TRIALS = None

base_config = {
    "xml_path": root / "assets/mjcf/so101_new_calib copy.xml",
    "root": root,
    "reward": make_default_reward(),
    "env_kwargs": {
        "camera_name": "gripper_cam",
        "render_profile": "train_fast",
        "video_render_profile": "standard",
        "video_camera_name": "overview_cam",
        "video_width": 320,
        "video_height": 240,
        "bounce_config": BounceConfig(success_bounces=5),
        "max_episode_steps": 256,
        "frame_skip": 20,
        "latent_steps": [],
    },
    "total_timesteps": 100_000,
    "nb_vec_env_train": 1,
    "nb_vec_env_test": 1,
    "n_eval_episodes": 3,
    "eval_max_steps": 256,
    "tensorboard_log": True,
    "print_spaces": False,
    "verbose": 0,
    "callback_verbose": 0,
    "device": "auto",
}
if ALGORITHM == "ppo":
    base_config.update(n_steps=256, batch_size=64, n_epochs=10)
else:
    base_config.update(buffer_size=3_000, learning_starts=1_000, batch_size=64)

spaces = {"ppo": PPO_GRID, "sac": SAC_GRID} if METHOD == "grid" else {"ppo": PPO_SPACE, "sac": SAC_SPACE}
search_space = copy.deepcopy(spaces[ALGORITHM])
# Example grid override: {"learning_rate": [1e-4, 3e-4], "batch_size": [64, 128]}
# Example continuous override: {"learning_rate": {"type": "float", "low": 1e-5, "high": 1e-3, "log": True}}

search_config = {
    "algorithm": ALGORITHM,
    "base_config": base_config,
    "method": METHOD,
    "study_name": STUDY_NAME,
    "search_space": search_space,
    "n_trials": N_TRIALS,
    "seed": 42,
    "eval_every_timesteps": 10_000,
    "pruning": False,
    "pruning_warmup_steps": 30_000,
    "n_startup_trials": 5,
}
print("Étude et modèles :", root / "models" / STUDY_NAME)
print("TensorBoard :", root / "runs" / f"{ALGORITHM}_bounce" / STUDY_NAME)


## Lancer ou continuer l’étude

Cette cellule lance réellement les entraînements. Adapter le budget et l’espace de recherche.
`pruning=True` permet d’arrêter des essais peu prometteurs après la phase initiale.
Un timeout, si ajouté, empêche de nouveaux essais après expiration mais n’interrompt pas l’essai en cours.


In [ ]:
study = optimize(**search_config)


## Résultats

`study.db` conserve l’étude, `trials.csv` récapitule les essais et `best_params.json`
contient les paramètres à réutiliser. `best_trial.json` indique le dossier du modèle sélectionné.
Le score concerne la dernière politique mise à jour, sauvegardée dans le checkpoint `*_last.zip`.
Le checkpoint `*_best.zip` conserve le classement habituel selon les métriques de jonglage.
Les changements de tâche ou de budget demandent un nouveau nom d’étude.


In [ ]:
completed = [trial for trial in study.trials if trial.state.name == "COMPLETE"]
if completed:
    print("Meilleur score :", study.best_value)
    display(JSON(study.best_params))
    print("Modèle final de cet essai :", study.best_trial.user_attrs["model_dir"])
else:
    print("Aucun essai terminé. Consulter les états dans trials.csv.")


## Réentraîner avec les meilleurs paramètres (facultatif)

Ce lancement utilise un modèle neuf, un nouveau dossier daté et un budget plus long.
Vérifier les configurations finalistes avec plusieurs seeds avant de conclure.
La seed de l’évaluation reste contrôlée mais l’état initial actuel est fixe, ce qui limite
la mesure de robustesse. Le notebook ne change pas les conditions de rebond ou la physique.


In [ ]:
RUN_FINAL_TRAINING = False
if RUN_FINAL_TRAINING:
    from so101_robotic_learning.bounce_rl.rl import train_ppo, train_sac
    final_config = copy.deepcopy(base_config)
    final_config.update(study.best_params)
    final_config.update(
        total_timesteps=500_000,
        seed=43,
        resume_from=None,
        model_dir_name=f"models/{ALGORITHM}_final_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"),
        save_eval_video=True,
        train_render_mode=None,
        eval_render_mode=None,
    )
    if ALGORITHM == "sac":
        final_config["save_replay_buffer"] = True
    trainer = train_ppo if ALGORITHM == "ppo" else train_sac
    trainer.train(**final_config)
